# Oso (SR530) landslide: lidar change detection and sediment budget

Pre-event lidar (PSLC, 2013) against post-event lidar (WSDOT, spring 2014) of the 22 March 2014 SR530 landslide
near Oso, Washington. This notebook runs the workflow of the `oso_cd` package step by step; every parameter is in
`config/oso.toml`. Data sources, licences and references: see `README.md` and `data/README.md`.

Data: Puget Sound LiDAR Consortium (2013 Tulalip Partnership, public domain) and Washington State Department of
Transportation (2014 Oso / Stillaguamish lidar by Quantum Spatial, freely available through PSLC). The raw lidar is
not redistributed in this repository.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

from oso_cd import pipeline

CONFIG = Path("..") / "config" / "oso.toml"
cfg = pipeline.load_config(CONFIG)
t0, t1 = cfg["labels"]["t0"], cfg["labels"]["t1"]

## 0. Units: the tiles say metres but hold feet

Both epochs carry EPSG:32148 (NAD83 / Washington North, *metres*) in their GeoKeys, but X, Y and Z are in US survey
feet (the 2014 vendor report says so, and the Z range of 235-1,070 matches the valley floor at Oso only in feet).
Read as labelled, vertical change would be ×3.28 too large and volumes ×35.3, and a check on the EPSG code would
pass. The PDAL pipeline overrides the CRS with NAD83 / Washington North (ftUS) + NAVD88 (ftUS), converts to metres on
the same datum and keeps the ground class (ASPRS 2).

In [ ]:
ground = pipeline.step_prepare(cfg)
# The pipeline as run (stage types and CRS only: the recorded JSON holds local file paths)
stages = json.loads((cfg["paths"]["work"] / f"{t1}_prepare_pipeline.json").read_text())["pipeline"]
for s in stages:
    print(s["type"], s.get("override_srs") or s.get("out_srs") or s.get("limits") or "")

## 1. Registration on stable terrain

The 2014 ground cloud is aligned to the 2013 one with py4dgeo's stable-area ICP: both clouds are thinned to one
point per 1 m voxel, supervoxels that changed less than 0.2 m are taken as stable, and a point-to-plane ICP is fitted
on them. At Oso the residual offset between the two surveys is a few centimetres.

In [ ]:
registered = pipeline.step_register(cfg, ground)
reg = json.loads((cfg["paths"]["results"] / "registration.json").read_text())
print("translation (m):", [round(v, 3) for v in reg["translation_m"]])

## 2. Ground DTMs (1 m) with a record of what is measured

Inverse-distance gridding of the ground points within res/√2 of each cell centre, then a GDAL inverse-distance fill
of the empty cells inside the common survey footprint. Under the forest, the 2013 survey has few ground returns, so a
large share of its cells is interpolated: each DTM comes with a source mask (measured / interpolated) and the
distance to the nearest measured cell.

In [ ]:
dtms = pipeline.step_dtm(cfg, {t0: ground[t0], t1: registered})
pd.DataFrame(json.loads((cfg["paths"]["results"] / "dtm_summary.json").read_text())["epochs"]).T

## 3. Error model and Level of Detection

On hand-drawn stable areas the true change is zero, so the raw DoD there is error. Its spread (NMAD) grows with
slope and is fitted as σ_DoD(θ) = √(a² + (b·tanθ)²); interpolated cells get an extra term that depends on the
distance to the nearest measured cell, and the spread of the per-area medians gives the systematic error σ_sys.
Each cell is significant when |ΔZ| > 1.96 · √(σ₀² + σ₁²) (95 %). Cells steeper than 45° have no σ and never count.

In [ ]:
change = pipeline.step_change(cfg, dtms)
em = change["error_model"]
print(f"a = {em['slope_model']['a_m']:.3f} m, b = {em['slope_model']['b_m']:.3f} m, "
      f"sigma_sys = {em['systematic']['sigma_sys_m']:.3f} m, median LoD = {em['lod_median_m']:.2f} m")
print(f"stable areas: NMAD measured {em['stable_measured_nmad_m']:.3f} m, "
      f"interpolated {em['stable_interpolated_nmad_m']:.3f} m, median offset {em['stable_measured_median_m']:+.3f} m")

## 4. Sediment budget by zone

Zones: the USGS approximate extent of the deposit (runout, Haugerud 2014) and the source area behind the headscarp.
Volumes are summed over significant cells only and split into cells measured in both epochs and cells interpolated
in at least one, because interpolation is the largest uncertainty after the zone outlines.

In [ ]:
display(change["volumes"].round(0))
display(change["uncertainty"].round(0))

## 5. Check against the reference run and figures

The same budget was first computed with the author's in-house toolbox (v0.4.1). This lean implementation must
reproduce its zonal net volumes within 2 %.

In [ ]:
validation = pipeline.step_validate(cfg, change["volumes"])
display(pd.DataFrame(validation["zones"]))
assert validation["passed"], "Zonal volumes differ from the reference run by more than the tolerance"
for path in pipeline.step_figures(cfg, change):
    display(Image(filename=str(path)))